In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
processed_path = r"processed_data"

In [ ]:


violence_path = os.path.join(processed_path, "V")
nonviolence_path = os.path.join(processed_path, "NV")

In [ ]:
from collections import defaultdict

video_frames = defaultdict(list)

for frame in os.listdir(violence_path):
    video_id = "_".join(frame.split("_")[1:3])
    video_frames[video_id].append(os.path.join(violence_path, frame))

for frame in os.listdir(nonviolence_path):
    video_id = "_".join(frame.split("_")[1:3]
                        )
    video_frames[video_id].append(os.path.join(nonviolence_path, frame))

print("Total videos:", len(video_frames))

Total videos: 2000


In [5]:
video_ids = list(video_frames.keys())

print("Example video IDs:", video_ids[:10])
print("Total videos:", len(video_ids))

Example video IDs: ['V_1000', 'V_100', 'V_101', 'V_102', 'V_103', 'V_104', 'V_105', 'V_106', 'V_107', 'V_108']
Total videos: 2000


In [6]:
labels = []

for vid in video_ids:
    if vid.startswith("V"):
        labels.append(1)
    else:
        labels.append(0)

print("Example labels:", labels[:10])

Example labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [7]:
from sklearn.model_selection import train_test_split

train_ids, test_ids, train_labels, test_labels = train_test_split(
    video_ids,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train videos:", len(train_ids))
print("Test videos:", len(test_ids))

Train videos: 1600
Test videos: 400


In [8]:
import cv2
import numpy as np
import tensorflow as tf

class VideoGenerator(tf.keras.utils.Sequence):

    def __init__(self, video_ids, labels, video_frames, batch_size=4):
        self.video_ids = video_ids
        self.labels = labels
        self.video_frames = video_frames
        self.batch_size = batch_size

    def __len__(self):
        return len(self.video_ids) // self.batch_size

    def __getitem__(self, idx):

        batch_ids = self.video_ids[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_labels = self.labels[idx*self.batch_size:(idx+1)*self.batch_size]

        X = []
        y = []

        for vid, label in zip(batch_ids, batch_labels):

            frames = sorted(self.video_frames[vid])

            sequence = []

            for frame in frames:
                img = cv2.imread(frame)
                img = cv2.resize(img,(160,160))
                img = img / 255.0
                sequence.append(img)

            X.append(sequence)
            y.append(label)

        return np.array(X), np.array(y)

In [9]:
train_gen = VideoGenerator(train_ids, train_labels, video_frames, batch_size=4)
test_gen = VideoGenerator(test_ids, test_labels, video_frames, batch_size=4)

In [10]:
X, y = train_gen[0]
print(X.shape, y.shape)

(4, 16, 160, 160, 3) (4,)


In [11]:
#MODEL BUILDING

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

# CNN feature extractor (applied to each frame)
model.add(TimeDistributed(
    Conv2D(32, (3,3), activation='relu'),
    input_shape=(16,160,160,3)
))
model.add(TimeDistributed(MaxPooling2D((2,2))))

model.add(TimeDistributed(Conv2D(64,(3,3),activation='relu')))
model.add(TimeDistributed(MaxPooling2D((2,2))))

model.add(TimeDistributed(Conv2D(128,(3,3),activation='relu')))
model.add(TimeDistributed(MaxPooling2D((2,2))))

model.add(TimeDistributed(Flatten()))

# Temporal learning
model.add(LSTM(64))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 16, 158, 158, 32)  896      
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 16, 79, 79, 32)   0         
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 16, 77, 77, 64)   18496     
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 16, 38, 38, 64)   0         
 tributed)                                                       
                                                                 
 time_distributed_4 (TimeDis  (None, 16, 36, 36, 128)  7

In [13]:
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=50
)

Epoch 1/50
400/400 [==============================] - 111s 202ms/step - loss: 0.6611 - accuracy: 0.5881 - val_loss: 0.5107 - val_accuracy: 0.7900
Epoch 2/50
400/400 [==============================] - 52s 129ms/step - loss: 0.4971 - accuracy: 0.7688 - val_loss: 0.5171 - val_accuracy: 0.7450
Epoch 3/50
400/400 [==============================] - 53s 132ms/step - loss: 0.3871 - accuracy: 0.8419 - val_loss: 0.3183 - val_accuracy: 0.8650
Epoch 4/50
400/400 [==============================] - 67s 168ms/step - loss: 0.2674 - accuracy: 0.8988 - val_loss: 0.2955 - val_accuracy: 0.8725
Epoch 5/50
400/400 [==============================] - 47s 119ms/step - loss: 0.2112 - accuracy: 0.9212 - val_loss: 0.2323 - val_accuracy: 0.9050
Epoch 6/50
400/400 [==============================] - 48s 119ms/step - loss: 0.1734 - accuracy: 0.9406 - val_loss: 0.2282 - val_accuracy: 0.9250
Epoch 7/50
400/400 [==============================] - 48s 119ms/step - loss: 0.1228 - accuracy: 0.9600 - val_loss: 0.2200 - val_a

In [14]:
model.save("violence_detection_model.h5")

In [1]:
from tensorflow.keras.models import load_model

model = load_model(r"C:\Violence-detection\models\violence_detection_model.h5")

print("Model loaded successfully")

Model loaded successfully


In [2]:
import pickle

with open("training_history.pkl", "wb") as f:
    pickle.dump(history.history, f)

NameError: name 'history' is not defined